# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and the main Dataset object
dataset = mlc.Dataset(url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Display some general properties
print(f"Dataset Identifier: {getattr(dataset.metadata, 'identifier', None)}")
print(f"Date Published: {getattr(dataset.metadata, 'datePublished', None)}")
print(f"Version: {getattr(dataset.metadata, 'version', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields
print('Record sets (by @id):')
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"  - @id: {rs['@id']} | name: {rs.get('name', '')}")
    record_sets.append(rs['@id'])
    print('    Fields:')
    for field in rs.get('fields', []):
        print(f"      - @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")

# If there are no field lists above, show record set info using mlcroissant
if not record_sets:
    # Try to infer record sets the mlcroissant way
    print('\nNo explicit record sets found in metadata. Using dataset.record_sets:')
    for recset_id in dataset.record_sets:
        print(f"  - @id: {recset_id}")
        record_sets.append(recset_id)
print(f"\nAvailable Record Sets: {record_sets}")

# Print a preview of the first few records for the first record set
if record_sets:
    first_rs = record_sets[0]
    print(f"\nSample records from record set '@id': {first_rs}")
    for i, record in enumerate(dataset.records(record_set=first_rs)):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Load all records from each record set into pandas DataFrames
# You may need to adjust the list below depending on the available record_set @ids.

# Use detected or known record set IDs
record_sets = record_sets  # from previous cell
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")

# Select the first available record set for further analysis
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]  # Use the first non-empty one
    print(f"Proceeding with record set: {selected_record_set_id}")
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Choose fields for numeric analysis
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    print(f"Available columns: {df.columns.tolist()}")

    # Try to pick a numeric field by common names
    possible_numeric = [col for col in df.columns if any(s in col.lower() for s in ['mw', 'mass', 'count', 'number', 'abundance', 'coverage', 'peptide'])]
    print(f"Possible numeric fields: {possible_numeric}")
    if possible_numeric:
        numeric_field = possible_numeric[0]
        print(f"Analyzing numeric field: {numeric_field}")
        # Remove missing/invalid and convert to float
        df = df.copy()
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field (e.g., 'description', 'accession', etc.)
        possible_groups = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if possible_groups:
            group_field = possible_groups[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            group_field = None
    else:
        numeric_field = None
        print("No numeric fields were detected for further analysis.")
        group_field = None
else:
    df = None
    numeric_field = None
    group_field = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Relationship between group and numeric field, if group field exists
    if group_field is not None:
        plt.figure(figsize=(12, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field or dataframe available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the [FAIR^2 Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library, explored available record sets and fields by their `@id`, and performed example filtering, normalization, and visualization.

We demonstrated how to reference fields by `@id` (column name) and apply typical EDA steps, such as filtering by a numeric threshold, normalization, and grouping. For further analysis, you may examine other record sets, relationships between fields, or export processed DataFrames for machine learning or downstream bioinformatics analyses.